# TP-2: CleaningData - Applying OOP and Design Patterns

## Exercise 4: Implement Factory Pattern for Data Transformations

**Objective:**
- Practice the **Factory Pattern** for scalable creation of transformation objects.
- Apply **abstraction** and **composition**.

**Concept:** Instead of manually creating objects, we use a **factory** that decides which transformation to create.

**Instructions:**
1. Create an abstract class `DataTransform` with a method `apply(df)`.
2. Implement subclasses: `NormalizeColumns`, `RemoveDuplicates`, `StandardizeText`.
3. Create a `TransformFactory` that returns transformation objects based on string input.

In [1]:
import os
from abc import ABC, abstractmethod

import pandas as pd

### Reused classes (from Exercises 1 and 2)
`CSVReader` to load the data, and the missing-value strategies to fill `NaN` before transforming.

In [2]:
class CSVReader:
    """Encapsulates reading and previewing a CSV file."""

    def __init__(self, file_path):
        self.file_path = file_path
        self._data = None  # private: loaded DataFrame

    def read(self):
        """Load the CSV file into a pandas DataFrame and return it."""
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")
        self._data = pd.read_csv(self.file_path)
        return self._data

    def preview(self, n=5):
        """Print the top n rows of the data."""
        if self._data is None:
            self.read()
        print(f"Top {n} rows of '{self.file_path}':")
        print(self._data.head(n))

    @property
    def data(self):
        """Read-only access to the loaded DataFrame."""
        return self._data

    def __repr__(self):
        status = "loaded" if self._data is not None else "not loaded"
        return f"CSVReader(file_path='{self.file_path}', status={status})"

In [3]:
class MissingValueStrategy(ABC):
    """Interface: every strategy must implement handle(df)."""

    @abstractmethod
    def handle(self, df):
        """Return a new DataFrame with missing values handled."""
        pass

class DropMissing(MissingValueStrategy):
    """Remove every row that has at least one missing value."""

    def handle(self, df):
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    """Fill missing values in numeric columns with the column mean."""

    def handle(self, df):
        df = df.copy()
        num_cols = df.select_dtypes(include="number").columns
        df[num_cols] = df[num_cols].fillna(df[num_cols].mean())
        return df


class FillMode(MissingValueStrategy):
    """Fill missing values in every column with its most frequent value."""

    def handle(self, df):
        df = df.copy()
        for col in df.columns:
            if df[col].isnull().any():
                df[col] = df[col].fillna(df[col].mode()[0])
        return df


class DataCleaner:
    """Context class: applies whichever strategy it is given."""

    def __init__(self, strategy):
        self._strategy = strategy

    def set_strategy(self, strategy):
        """Switch to another strategy at runtime."""
        self._strategy = strategy

    def clean(self, df):
        print(f"Applying strategy: {type(self._strategy).__name__}")
        return self._strategy.handle(df)

### Step 1: Abstract class `DataTransform` (abstraction)
Every transformation must implement `apply(df)` and return a new DataFrame.

In [4]:
class DataTransform(ABC):
    """Interface: every transformation must implement apply(df)."""

    @abstractmethod
    def apply(self, df):
        """Return a new, transformed DataFrame."""
        pass

### Step 2: Concrete transformations

| Class | What it does |
|---|---|
| `NormalizeColumns` | Min-max scaling: `(x - min) / (max - min)` → values between 0 and 1 |
| `RemoveDuplicates` | Removes duplicate rows |
| `StandardizeText` | Removes extra spaces and converts text to Title Case (`"  new york "` → `"New York"`) |

In [5]:
class NormalizeColumns(DataTransform):
    """Min-max scale numeric columns to the range [0, 1]."""

    def __init__(self, columns=None):
        self.columns = columns  # None = all numeric columns

    def apply(self, df):
        df = df.copy()
        cols = self.columns or df.select_dtypes(include="number").columns.tolist()
        for col in cols:
            col_min, col_max = df[col].min(), df[col].max()
            if col_max == col_min:
                df[col] = 0.0
            else:
                df[col] = (df[col] - col_min) / (col_max - col_min)
        return df


class RemoveDuplicates(DataTransform):
    """Drop rows that are exact duplicates of a previous row."""

    def apply(self, df):
        before = len(df)
        df = df.drop_duplicates().reset_index(drop=True)
        print(f"Removed {before - len(df)} duplicate row(s)")
        return df


class StandardizeText(DataTransform):
    """Strip extra spaces and convert text columns to Title Case."""

    def apply(self, df):
        df = df.copy()
        text_cols = df.select_dtypes(exclude="number").columns
        for col in text_cols:
            df[col] = df[col].str.strip().str.title()
        return df

### Step 3: `TransformFactory`
- A dictionary maps a **string name** to a **class**.
- `create(name)` returns a new object of the right class - the user never writes `NormalizeColumns()` directly.
- **Scalable**: to add a new transformation, we only add one line to the dictionary.

In [6]:
class TransformFactory:
    """Factory: creates a transformation object from a string name."""

    _transforms = {
        "normalize": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }

    @classmethod
    def create(cls, name, **kwargs):
        key = name.strip().lower()
        if key not in cls._transforms:
            raise ValueError(f"Unknown transform '{name}'. Available: {cls.available()}")
        return cls._transforms[key](**kwargs)

    @classmethod
    def available(cls):
        return list(cls._transforms)

### Test 1: Create objects from strings

In [7]:
print("Available transforms:", TransformFactory.available())

for name in TransformFactory.available():
    obj = TransformFactory.create(name)
    print(f"'{name}' -> {type(obj).__name__}")

Available transforms: ['normalize', 'remove_duplicates', 'standardize_text']
'normalize' -> NormalizeColumns
'remove_duplicates' -> RemoveDuplicates
'standardize_text' -> StandardizeText


In [8]:
# Unknown name gives a clear error
try:
    TransformFactory.create("sort_rows")
except ValueError as e:
    print("Error:", e)

Error: Unknown transform 'sort_rows'. Available: ['normalize', 'remove_duplicates', 'standardize_text']


### Test 2: Messy example data
`sample_data.csv` has no duplicates or messy text, so we first test on a small messy DataFrame.
Rows 0 and 1 are the **same person** written differently.

In [9]:
messy = pd.DataFrame({
    "name":  ["  alice", "ALICE ", "bob", "charlie"],
    "city":  ["new york", " New York", "los angeles", "CHICAGO "],
    "score": [85, 85, 90, 70],
})
messy

,name,city,score
0,alice,new york,85
1,ALICE,New York,85
2,bob,los angeles,90
3,charlie,CHICAGO,70


In [10]:
# Composition: apply several transformations one after another
df_t = messy
for name in ["standardize_text", "remove_duplicates", "normalize"]:
    df_t = TransformFactory.create(name).apply(df_t)
df_t

Removed 1 duplicate row(s)


,name,city,score
0,Alice,New York,0.75
1,Bob,Los Angeles,1.00
2,Charlie,Chicago,0.00


Order matters: after `standardize_text`, `"  alice"` and `"ALICE "` both become `"Alice"`, so `remove_duplicates` can detect them.

### Test 3: On `sample_data.csv`

In [11]:
df = CSVReader("sample_data.csv").read()
df = DataCleaner(FillMean()).clean(df)  # fill NaN first

steps = [
    TransformFactory.create("standardize_text"),
    TransformFactory.create("remove_duplicates"),
    TransformFactory.create("normalize", columns=["age", "height_cm", "weight_kg", "score"]),
]
for step in steps:
    df = step.apply(df)
df

Applying strategy: FillMean
Removed 0 duplicate row(s)


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,0.25000,0.333333,0.433333,New York,0.588235
1,2,Bob,0.40625,0.800000,0.500000,Los Angeles,0.882353
2,3,Charlie,1.00000,0.533333,0.566667,Chicago,0.647059
3,4,David,0.40625,0.466667,0.833333,Houston,0.000000
4,5,Eva,0.00000,0.000000,0.000000,New York,0.764706
5,6,Frank,0.37500,1.000000,1.000000,Los Angeles,0.647059
6,7,Grace,0.40625,0.133333,0.166667,Chicago,1.000000


### Summary
- **Abstraction**: `DataTransform` defines *what* a transformation does (`apply(df)`), subclasses define *how*.
- **Factory Pattern**: `TransformFactory.create("normalize")` creates the object from a string, so the calling code doesn't depend on concrete classes.
- **Composition**: a list of transformation objects is applied one after another to build a bigger process.
- Extra arguments can be passed through the factory, e.g. `create("normalize", columns=[...])` (we keep `id` unchanged).